# 01 — Generate Tool-Calling Training Data

**CPU only. Run in Modal notebook before notebook 02.**

Creates ~400 synthetic training sequences that teach Moshi:
1. Emit `<|tool_call|>get_time<|tool_end|>` when context implies a time query
2. Emit `<|tool_call|>get_weather CITY<|tool_end|>` for weather
3. Respond naturally after a silent `<|tool_result|>…<|tool_result_end|>` injection

Output saved to `data/tool_calls.jsonl`.

In [ ]:
# Install / upgrade if needed (comment out after first run)
!pip install sentencepiece huggingface_hub -q

In [ ]:
import json, random, sys, os
from pathlib import Path

# ── Config — fill these in ────────────────────────────────────────────────────
HF_PATCHED_REPO = "abrarfahim/moshi-tool-patched"
HF_TOKEN        = os.environ.get("HF_TOKEN", None)  # or paste token string here
# ─────────────────────────────────────────────────────────────────────────────

# Clone repo if not present (Modal environment)
REPO = Path("/repo")
if not REPO.exists():
    import subprocess
    subprocess.run(["git", "clone",
        "https://github.com/syedfahimabrar/moshi-D-gu.git",
        str(REPO)], check=True)

sys.path.insert(0, str(REPO / "moshi"))

import sentencepiece as spm

TOKENIZER_PATH = REPO / "weights/patched/tokenizer_spm_32k_3.model"
if not TOKENIZER_PATH.exists():
    from huggingface_hub import hf_hub_download
    TOKENIZER_PATH = Path(hf_hub_download(
        repo_id=HF_PATCHED_REPO, filename="tokenizer_spm_32k_3.model",
        local_dir="/tmp/patched", token=HF_TOKEN))

tok = spm.SentencePieceProcessor()
tok.Load(str(TOKENIZER_PATH))
print(f"Tokenizer: {tok.get_piece_size()} tokens")

In [ ]:
PAD_ID             = 3
TOOL_CALL_ID       = 32000
TOOL_END_ID        = 32001
TOOL_RESULT_ID     = 32002
TOOL_RESULT_END_ID = 32003

assert tok.id_to_piece(TOOL_CALL_ID) == "<|tool_call|>", "Tokenizer not patched!"
print("Special tokens:", [tok.id_to_piece(i) for i in [32000,32001,32002,32003]])

In [ ]:
def encode(text):
    return [i for i in tok.encode(text) if i < 32000]

_DAYS   = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
_MONTHS = ["January","February","March","April","May","June",
           "July","August","September","October","November","December"]

_TIME_TRIGGERS = [
    "what time is it", "tell me the time", "what's the current time",
    "do you know what time it is", "can you check the time",
    "what time do you have", "I need to know the time",
    "checking the time", "let me check the time",
]

_TIME_RESPONSES = [
    "It's {t} right now.", "The time is {t}.", "Right now it's {t}.",
    "It's currently {t}.", "The current time is {t}.",
]

def _random_time():
    h, m = random.randint(1,12), random.randint(0,59)
    ap = random.choice(["AM","PM"])
    result = f"Live time: {h}:{m:02d} {ap} on {random.choice(_DAYS)}, {random.choice(_MONTHS)} {random.randint(1,28)}, 2026"
    spoken = (f"{h} o'clock {ap.lower()}" if m==0 else
              f"quarter past {h} {ap.lower()}" if m==15 else
              f"half past {h} {ap.lower()}" if m==30 else
              f"{h}:{m:02d} {ap.lower()}")
    return result, spoken

def make_time_example():
    result_str, spoken = _random_time()
    trigger_ids = encode(random.choice(_TIME_TRIGGERS))
    gap = [PAD_ID] * random.randint(5, 20)
    tool_seq = ([TOOL_CALL_ID] + encode("get_time") + [TOOL_END_ID] +
                [TOOL_RESULT_ID] + encode(result_str) + [TOOL_RESULT_END_ID] +
                encode(random.choice(_TIME_RESPONSES).format(t=spoken)))
    tokens = trigger_ids + gap + tool_seq
    mask   = [0]*(len(trigger_ids)+len(gap)) + [1]*len(tool_seq)
    return {"tokens": tokens, "mask": mask, "type": "time"}

print("Time example:", tok.decode([t for t in make_time_example()["tokens"] if t < 32000]))

In [ ]:
_CITIES = [
    "London","New York","Tokyo","Paris","Sydney","Dubai","Mumbai","Toronto",
    "Berlin","Singapore","Los Angeles","Seoul","Bangkok","Istanbul","Cairo",
    "Amsterdam","Madrid","Rome","Hong Kong","Kuala Lumpur","Dhaka","Karachi",
]

_CONDITIONS = [
    ("sunny",         lambda c: f"Live weather: {c}: ☀️ {random.randint(18,35)}°C"),
    ("cloudy",        lambda c: f"Live weather: {c}: ☁️ {random.randint(10,22)}°C"),
    ("partly cloudy", lambda c: f"Live weather: {c}: ⛅ {random.randint(12,28)}°C"),
    ("rainy",         lambda c: f"Live weather: {c}: 🌧️ {random.randint(8,18)}°C"),
    ("snowy",         lambda c: f"Live weather: {c}: ❄️ {random.randint(-8,2)}°C"),
    ("windy",         lambda c: f"Live weather: {c}: 💨 {random.randint(10,20)}°C"),
]

_WEATHER_RESPONSES = {
    "sunny":         "It's sunny and warm in {city}, {temp} right now.",
    "cloudy":        "Overcast in {city} today, about {temp}.",
    "partly cloudy": "Partly cloudy in {city}, {temp}.",
    "rainy":         "It's raining in {city}, {temp} out there.",
    "snowy":         "Snowing in {city} right now, {temp}.",
    "windy":         "Pretty windy in {city}, {temp}.",
}

def make_weather_example(city=None):
    city = city or random.choice(_CITIES)
    cond, result_fn = random.choice(_CONDITIONS)
    result_str = result_fn(city)
    temp = result_str.split("°C")[0].split()[-1] + "°C"
    triggers = [f"what's the weather in {city}", f"weather in {city}",
                f"how's the weather in {city}", f"check weather in {city}"]
    trigger_ids = encode(random.choice(triggers))
    gap = [PAD_ID] * random.randint(5, 20)
    tool_seq = ([TOOL_CALL_ID] + encode(f"get_weather {city}") + [TOOL_END_ID] +
                [TOOL_RESULT_ID] + encode(result_str) + [TOOL_RESULT_END_ID] +
                encode(_WEATHER_RESPONSES[cond].format(city=city, temp=temp)))
    tokens = trigger_ids + gap + tool_seq
    mask   = [0]*(len(trigger_ids)+len(gap)) + [1]*len(tool_seq)
    return {"tokens": tokens, "mask": mask, "type": "weather"}

print("Weather example:", tok.decode([t for t in make_weather_example("London")["tokens"] if t < 32000]))

In [ ]:
_NULL_PHRASES = [
    "hey how are you doing", "tell me a fun fact", "what do you think about music",
    "how does the internet work", "what's your favorite book",
    "what are some good movies", "tell me about space exploration",
    "what is machine learning", "describe a beautiful sunset",
]

def make_null_example():
    phrase_ids   = encode(random.choice(_NULL_PHRASES))
    gap          = [PAD_ID] * random.randint(5, 20)
    response_ids = encode(random.choice(["I'd be happy to help.", "Great question.",
                                         "That's interesting.", "Let me think about that."]))
    tokens = phrase_ids + gap + response_ids
    mask   = [0]*(len(phrase_ids)+len(gap)) + [1]*len(response_ids)
    return {"tokens": tokens, "mask": mask, "type": "null"}

In [ ]:
random.seed(42)
dataset = []

for _ in range(150):
    dataset.append(make_time_example())

for city in _CITIES:
    dataset.append(make_weather_example(city))
    dataset.append(make_weather_example(city))
for _ in range(140):
    dataset.append(make_weather_example())

for _ in range(50):
    dataset.append(make_null_example())

random.shuffle(dataset)

counts = {}
for ex in dataset:
    counts[ex["type"]] = counts.get(ex["type"], 0) + 1

print(f"Total: {len(dataset)} examples")
print("Breakdown:", counts)
print(f"Avg length: {sum(len(e['tokens']) for e in dataset)/len(dataset):.0f} tokens")

In [ ]:
OUT = Path("data/tool_calls.jsonl")
OUT.parent.mkdir(exist_ok=True)

with OUT.open("w") as f:
    for ex in dataset:
        f.write(json.dumps(ex) + "\n")

print(f"Saved {len(dataset)} examples → {OUT}")

In [ ]:
# Preview one of each type
for t in ["time", "weather", "null"]:
    ex = next(e for e in dataset if e["type"] == t)
    readable = []
    for tid in ex["tokens"]:
        if   tid == TOOL_CALL_ID:       readable.append("<CALL>")
        elif tid == TOOL_END_ID:        readable.append("<END>")
        elif tid == TOOL_RESULT_ID:     readable.append("<RESULT>")
        elif tid == TOOL_RESULT_END_ID: readable.append("<RESULT_END>")
        elif tid == PAD_ID:             readable.append("[P]")
        else: readable.append(tok.id_to_piece(tid).replace("▁", " "))
    print(f"[{t}] {''.join(readable[:80])}...")
    print()